In [21]:
try:
    !pip install -q \
    chromadb \
    wikipedia-api \
    langchain-text-splitters \
    google-genai

    print("Libraries installed successfully!")

except Exception as e:
    print("Installation Error:")
    print(e)

Libraries installed successfully!


In [22]:
try:
    import wikipediaapi
    import os

    wiki = wikipediaapi.Wikipedia(
        language="en",
        user_agent="networking-rag-project"
    )

    topics = [
        "Transmission Control Protocol",
        "User Datagram Protocol",
        "Domain Name System",
        "Internet Protocol",
        "OSI model",
        "TCP/IP model",
        "Routing",
        "Network switch",
        "Computer network",
        "Hypertext Transfer Protocol"
    ]

    os.makedirs("networking_docs", exist_ok=True)

    for topic in topics:

        page = wiki.page(topic)

        if page.exists():

            filename = (
                topic.replace("/", "_")
                     .replace(" ", "_")
            ) + ".txt"

            with open(
                f"networking_docs/{filename}",
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page.text)

            print(f"Saved: {filename}")

        else:
            print(f"Page not found: {topic}")

    print("\nAll documents downloaded successfully!")

except Exception as e:
    print("Download Error:")
    print(e)

Saved: Transmission_Control_Protocol.txt
Saved: User_Datagram_Protocol.txt
Saved: Domain_Name_System.txt
Saved: Internet_Protocol.txt
Saved: OSI_model.txt
Saved: TCP_IP_model.txt
Saved: Routing.txt
Saved: Network_switch.txt
Saved: Computer_network.txt
Saved: Hypertext_Transfer_Protocol.txt

All documents downloaded successfully!


In [23]:
try:
    import os

    documents = []

    folder_path = "networking_docs"

    for file in os.listdir(folder_path):

        if file.endswith(".txt"):

            file_path = os.path.join(folder_path, file)

            with open(file_path, "r", encoding="utf-8") as f:
                documents.append(f.read())

    print("Documents loaded successfully!")
    print("Number of documents:", len(documents))

except Exception as e:
    print("Loading Error:")
    print(e)

Documents loaded successfully!
Number of documents: 10


In [24]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2500,
        chunk_overlap=50
    )

    chunks = []

    for document in documents:

        document_chunks = splitter.split_text(document)

        chunks.extend(document_chunks)

    print("Chunking completed successfully!")
    print("Total Chunks:", len(chunks))

except Exception as e:
    print("Chunking Error:")
    print(e)

Chunking completed successfully!
Total Chunks: 182


In [25]:
try:
    from google import genai
    from google.colab import userdata
    import time

    client = genai.Client(
        api_key=userdata.get("GEMINI_API_KEY")
    )

    chunk_embeddings = []

    for i, chunk in enumerate(chunks):

        response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=chunk
        )

        chunk_embeddings.append(
            response.embeddings[0].values
        )

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1} chunks")

        time.sleep(1)

    print("Embedding generation completed!")
    print("Total embeddings:", len(chunk_embeddings))

except Exception as e:
    print("Embedding Error:")
    print(e)

Processed 10 chunks
Processed 20 chunks
Processed 30 chunks
Processed 40 chunks
Processed 50 chunks
Processed 60 chunks
Processed 70 chunks
Processed 80 chunks
Processed 90 chunks
Processed 100 chunks
Processed 110 chunks
Processed 120 chunks
Processed 130 chunks
Processed 140 chunks
Processed 150 chunks
Processed 160 chunks
Processed 170 chunks
Processed 180 chunks
Embedding generation completed!
Total embeddings: 182


In [26]:
try:
    import chromadb

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb"
    )

    collection = chroma_client.get_or_create_collection(
        name="networking_docs"
    )

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_embeddings
    )

    print("Chunks and embeddings stored successfully!")
    print("Total records:", collection.count())

except Exception as e:
    print("ChromaDB Error:")
    print(e)

Chunks and embeddings stored successfully!
Total records: 182


In [27]:
try:

    print("Verifying ChromaDB storage...")

    print("Collection Name:", collection.name)
    print("Total Records:", collection.count())

except Exception as e:

    print("Verification Error:")
    print(e)

Verifying ChromaDB storage...
Collection Name: networking_docs
Total Records: 182


In [28]:
try:

    import pickle

    with open("chunks.pkl", "wb") as f:
        pickle.dump(chunks, f)

    with open("chunk_embeddings.pkl", "wb") as f:
        pickle.dump(chunk_embeddings, f)

    print("Files saved successfully!")

except Exception as e:

    print(e)

Files saved successfully!
